# NB_hospital_daily_summary

Builds `adb_db_caresync.gold.hospital_daily_summary` from the silver layer per the STTM (`Gold/sttm_silver_to_gold.xlsx`).

* **Grain:** one row per hospital per calendar day (days that have lab tests)
* **Driving table:** `silver.lab_results` grouped by `hospital_id`, `test_date`
* **Joins (LEFT):** `silver.hospitals` on `hospital_id`; `silver.calendar` on `date_key = yyyyMMdd(test_date)` (`date_key` is an int, so the formatted date is cast to int)
* **Window:** `LAG(total_tests)` by hospital ordered by `summary_date` (previous day with tests)
* **Load:** full rebuild with `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
%sql
CREATE OR REPLACE TABLE adb_db_caresync.gold.hospital_daily_summary AS
WITH daily AS (
  SELECT
    hospital_id,
    test_date                             AS summary_date,
    COUNT(*)                              AS total_tests,
    COUNT(DISTINCT patient_id)            AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)             AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count
  FROM adb_db_caresync.silver.lab_results
  GROUP BY hospital_id, test_date
),
trend AS (
  SELECT
    *,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily
)
SELECT
  t.hospital_id,
  h.hospital_name,
  t.summary_date,
  c.day_name,
  c.is_weekend,
  CAST(t.total_tests AS INT)             AS total_tests,
  CAST(t.unique_patients_tested AS INT)  AS unique_patients_tested,
  CAST(t.active_doctor_count AS INT)     AS active_doctor_count,
  CAST(t.abnormal_test_count AS INT)     AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)     AS critical_test_count,
  CAST(t.abnormal_test_count / t.total_tests * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(t.critical_test_count / t.total_tests * 100 AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(t.prev_day_total_tests AS INT)    AS prev_day_total_tests,
  CAST(
    CASE WHEN t.prev_day_total_tests IS NULL OR t.prev_day_total_tests = 0 THEN NULL
         ELSE (t.total_tests - t.prev_day_total_tests) / t.prev_day_total_tests * 100
    END AS DECIMAL(6,2))                 AS test_volume_change_pct
FROM trend t
LEFT JOIN adb_db_caresync.silver.hospitals h
  ON t.hospital_id = h.hospital_id
LEFT JOIN adb_db_caresync.silver.calendar c
  ON c.date_key = CAST(date_format(t.summary_date, 'yyyyMMdd') AS INT)

In [0]:
updated_df = spark.table("adb_db_caresync.gold.hospital_daily_summary")
record_count = updated_df.count()

dbutils.notebook.exit(str(record_count))